# 2026 FIFA World Cup — Notebook 2: Model Training
Train on 2010-2018 | Validate on 2022 | XGBoost + Poisson

In [1]:
import pandas as pd
import numpy as np
import warnings
import pickle
import requests
from io import StringIO
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, mean_absolute_error
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import PoissonRegressor
from xgboost import XGBClassifier
warnings.filterwarnings('ignore')
DATA_DIR = '../data'
NAME_MAP = {
    'USA':'United States','Ivory Coast':'Cote dIvoire',
    'Korea Republic':'South Korea',
    'Bosnia-Herzegovina':'Bosnia and Herzegovina',
    'Turkiye':'Turkey','Türkiye':'Turkey',
    'IR Iran':'Iran','Congo DR':'DR Congo',
    'Czechia':'Czech Republic',
}
team_features = pd.read_csv(f'{DATA_DIR}/team_features.csv')
xg_data = pd.read_csv(f'{DATA_DIR}/xg_data.csv')
print('Data loaded')
print(f'  team_features: {team_features.shape}')
print(f'  xg_data: {xg_data.shape}')

Data loaded
  team_features: (48, 13)
  xg_data: (128, 9)


## 1 · Load Historical Match Data

In [2]:
BASE_JF = 'https://raw.githubusercontent.com/jfjelstul/worldcup/master/data-csv'
def fetch_csv(url):
    r = requests.get(url, timeout=30)
    return pd.read_csv(StringIO(r.text))

print('Loading match data...')
matches_df  = pd.read_csv(f'{DATA_DIR}/matches.csv')
team_app_df = pd.read_csv(f'{DATA_DIR}/team_appearances.csv')
for df in [matches_df, team_app_df]:
    if 'match_date' in df.columns:
        df['match_date'] = pd.to_datetime(df['match_date'])
    for col in df.columns:
        if 'team' in col.lower():
            df[col] = df[col].replace(NAME_MAP)

mens_ids = matches_df[matches_df['tournament_name'].str.contains('Men',na=False)]['match_id'].unique()
mens = matches_df[
    matches_df['match_id'].isin(mens_ids) &
    (matches_df['match_date']>='2010-01-01') &
    (matches_df['match_date']<='2022-12-31')
].copy()
print(f'Matches loaded: {len(mens)}')
print(mens.groupby(mens['match_date'].dt.year)['match_id'].count())

Loading match data...
Matches loaded: 256
match_date
2010    64
2014    64
2018    64
2022    64
Name: match_id, dtype: int64


## 2 · Build Feature Matrix

In [3]:
feat_idx = team_features.set_index('team')
xg_lookup = {(r['home'],r['away']):(r['home_xg'],r['away_xg']) for _,r in xg_data.iterrows()}

def get_feat(team, col, default=0.0):
    try:
        val = feat_idx.loc[team, col]
        return val if not pd.isna(val) else default
    except:
        return default

def hist_stats(team, before_date, team_app_df):
    h = team_app_df[
        (team_app_df['team_name']==team) &
        (team_app_df['match_date']<before_date)
    ]
    if len(h)==0:
        rs = get_feat(team,'rank_score',0.4)
        rank = get_feat(team,'fifa_rank',50)
        return {
            'wr':max(0.1,0.5-(rank-1)*0.007),
            'avgf':max(0.5,1.8-(rank-1)*0.02),
            'avga':min(2.5,0.8+(rank-1)*0.02),
            'avgd':0.0
        }
    return {
        'wr':h['win'].mean(),
        'avgf':h['goals_for'].mean(),
        'avga':h['goals_against'].mean(),
        'avgd':h['goal_differential'].mean()
    }

# Tournament recency weights — recent matches matter more
WEIGHTS = {2010:0.6, 2011:0.6, 2012:0.65, 2013:0.65,
           2014:0.7, 2015:0.75, 2016:0.8, 2017:0.85,
           2018:0.9, 2019:0.9, 2020:0.95, 2021:0.95, 2022:1.0}

print('Engineering match features...')
rows = []
for _,m in mens.iterrows():
    ht = m['home_team_name']
    at = m['away_team_name']
    dt = m['match_date']
    yr = dt.year
    hs = hist_stats(ht, dt, team_app_df)
    as_ = hist_stats(at, dt, team_app_df)
    hr = get_feat(ht,'rank_score',0.4)
    ar = get_feat(at,'rank_score',0.4)
    xgp = xg_lookup.get((ht,at), xg_lookup.get((at,ht),None))
    if xgp:
        h_xg = xgp[0] if (ht,at) in xg_lookup else xgp[1]
        a_xg = xgp[1] if (ht,at) in xg_lookup else xgp[0]
    else:
        h_xg = a_xg = 0.0
    hg = m['home_team_score']
    ag = m['away_team_score']
    target = 0 if hg>ag else (1 if hg==ag else 2)
    rows.append({
        'date':dt,'year':yr,'home':ht,'away':at,
        'home_score':hg,'away_score':ag,
        'knockout':int(m['knockout_stage']),
        'rank_diff':hr-ar,
        'wr_diff':hs['wr']-as_['wr'],
        'gf_diff':hs['avgf']-as_['avgf'],
        'ga_diff':hs['avga']-as_['avga'],
        'gd_diff':hs['avgd']-as_['avgd'],
        'home_rank':hr,'away_rank':ar,
        'xg_diff':h_xg-a_xg,
        'weight':WEIGHTS.get(yr,0.7),
        'target':target
    })

match_data = pd.DataFrame(rows)
print(f'Match dataset: {len(match_data)} matches x {len(match_data.columns)} cols')
print('Outcomes:', dict(match_data['target'].value_counts()), '(0=HW,1=D,2=AW)')

Engineering match features...
Match dataset: 256 matches x 17 cols
Outcomes: {0: np.int64(106), 2: np.int64(93), 1: np.int64(57)} (0=HW,1=D,2=AW)


## 3 · Train / Validate Split

In [4]:
FEATS = ['rank_diff','wr_diff','gf_diff','ga_diff','gd_diff',
         'home_rank','away_rank','knockout','xg_diff']

train = match_data[match_data['year']<2022].dropna(subset=FEATS)
test  = match_data[match_data['year']==2022].dropna(subset=FEATS)

X_tr = train[FEATS].values
y_tr = train['target'].values
w_tr = train['weight'].values
X_te = test[FEATS].values
y_te = test['target'].values

print(f'Train: {len(train)} matches (2010-2018)')
print(f'Test:  {len(test)} matches (2022 WC — held out)')

Train: 192 matches (2010-2018)
Test:  64 matches (2022 WC — held out)


## 4 · Train XGBoost Classifier

In [5]:
xgb = XGBClassifier(
    n_estimators=400, max_depth=4, learning_rate=0.04,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='mlogloss', random_state=42, n_jobs=-1
)
xgb.fit(X_tr, y_tr, sample_weight=w_tr,
        eval_set=[(X_te,y_te)], verbose=False)

clf = CalibratedClassifierCV(xgb, cv='prefit', method='isotonic')
clf.fit(X_tr, y_tr, sample_weight=w_tr)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_acc = cross_val_score(xgb, X_tr, y_tr, cv=cv,
                         scoring='accuracy')
test_acc = accuracy_score(y_te, clf.predict(X_te))

print(f'5-Fold CV Accuracy (train): {cv_acc.mean():.3f} +/- {cv_acc.std():.3f}')
print(f'Test Accuracy (2022 WC):    {test_acc:.3f}')
print(f'(Random baseline=33% | Good models reach 50-55%)')
print()
print(classification_report(y_te, clf.predict(X_te),
      target_names=['Home Win','Draw','Away Win']))

5-Fold CV Accuracy (train): 0.474 +/- 0.073
Test Accuracy (2022 WC):    0.562
(Random baseline=33% | Good models reach 50-55%)

              precision    recall  f1-score   support

    Home Win       0.70      0.66      0.68        29
        Draw       0.50      0.27      0.35        15
    Away Win       0.45      0.65      0.53        20

    accuracy                           0.56        64
   macro avg       0.55      0.52      0.52        64
weighted avg       0.58      0.56      0.55        64



## 5 · Poisson Scoreline Model

In [6]:
P_FEATS = ['rank_diff','gf_diff','ga_diff','home_rank','away_rank','xg_diff']

pr_home = PoissonRegressor(alpha=1.0, max_iter=500)
pr_home.fit(train[P_FEATS], train['home_score'],
            sample_weight=train['weight'])

away_train = train[P_FEATS].copy()
away_train['rank_diff'] = away_train['rank_diff'] * -1
away_train['xg_diff']   = away_train['xg_diff']   * -1
pr_away = PoissonRegressor(alpha=1.0, max_iter=500)
pr_away.fit(away_train, train['away_score'],
            sample_weight=train['weight'])

pred_h = pr_home.predict(test[P_FEATS])
away_te = test[P_FEATS].copy()
away_te['rank_diff'] *= -1
away_te['xg_diff']   *= -1
pred_a = pr_away.predict(away_te)
mae = (mean_absolute_error(test['home_score'],pred_h) +
       mean_absolute_error(test['away_score'],pred_a)) / 2
print(f'Poisson MAE: {mae:.3f} goals average error per team')
print(f'(Means predictions are on average within {mae:.2f} goals of actual)')

Poisson MAE: 0.955 goals average error per team
(Means predictions are on average within 0.96 goals of actual)


## 6 · Save Model

In [7]:
with open(f'{DATA_DIR}/model.pkl','wb') as f:
    pickle.dump({
        'classifier': clf,
        'poisson_home': pr_home,
        'poisson_away': pr_away,
        'features': FEATS,
        'poisson_features': P_FEATS,
        'test_accuracy': float(test_acc),
        'cv_accuracy': float(cv_acc.mean()),
        'mae': float(mae),
    }, f)
match_data.to_csv(f'{DATA_DIR}/match_dataset.csv', index=False)
print(f'Model saved — accuracy: {test_acc:.3f} | MAE: {mae:.3f}')
print('Notebook 2 complete!')

Model saved — accuracy: 0.562 | MAE: 0.955
Notebook 2 complete!
